# CPE342 Task 4 (Round 43): Zero-Crop RoPE Multi-Resolution TTA (`dinov3Hp_384` @ `160×304`, `192×384`, `224×432`)

**Self-contained notebook** (uses the restored `t4_dinov3Hp_384_fold0..4.pt` and `t4_dinov3Hp_384_full.pt` checkpoints on Drive + `t4_data.zip` — no `cpe342_code.zip` re-upload required).

### Protocol
- Loads each fold's EMA checkpoint with `timm.create_model("vit_huge_plus_patch16_dinov3.lvd1689m", num_classes=5, dynamic_img_size=True)`.
- Predicts OOF validation images (strictly training folds) under `full`, `zoom70`, `centre_grey` at three native 2:1 RoPE grids:
  - `160×304` (`10×19` patches)
  - `192×384` (`12×24` patches — baseline, checked for `mean|diff| < 2e-3`, `argmax agree >= 99.8%`, `max|diff| < 0.08` vs cached `r30_views`)
  - `224×432` (`14×27` patches)
- Saves tiny outputs (< 5 MB total) to `Drive/out/preds/`:
  - `t4_mres_dinov3Hp_384_fold{0..4}.npz` (used on laptop for strict Nested 5-fold `--eval` against `v38`)
  - `t4_mres_dinov3Hp_384_test.npz` (if `EXTRACT_TEST = True`: inference-only cache for `fold0..4` + `full`, never read by `--eval`)

In [ ]:
# ---- Settings -------------------------------------------------------------------------------------------------------------
MODEL = "vit_huge_plus_patch16_dinov3.lvd1689m"
TAG = "dinov3Hp_384"
BASE_HW = (192, 384)
RESOLUTIONS = [(160, 304), (192, 384), (224, 432)]
RES_KEYS = [f"{h}x{w}" for h, w in RESOLUTIONS]
EVAL_VIEWS = ["full", "zoom70", "centre_grey"]
FOLDS = [0, 1, 2, 3, 4]
BS = 64
EXTRACT_TEST = True       # True -> also score test.csv across fold0..4.pt + full.pt (inference-only cache)
AUTO_DISCONNECT = True    # end runtime 60 s after completion or on error to save Colab CU
DRIVE_CANDIDATES = [
    "/content/drive/MyDrive/cpe342_colab",
    "/content/drive/MyDrive/cpe342-colab",
]
try:
    _ip = get_ipython()
except NameError:
    _ip = None
if AUTO_DISCONNECT and _ip is not None:
    def _end_runtime_on_error(shell, etype, evalue, tb, tb_offset=None):
        shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
        import time as _t
        print(f"\nerror -> ending runtime in 60 s [{_t.strftime('%H:%M')} UTC]", flush=True)
        _t.sleep(60)
        from google.colab import runtime
        runtime.unassign()
    _ip.set_custom_exc((Exception,), _end_runtime_on_error)

In [ ]:
# ---- Packages & GPU check -------------------------------------------------------------------------------------------------
!pip -q install timm==1.0.30
import numpy, pandas, sklearn, timm, torch, torchvision

print("torch", torch.__version__, "| timm", timm.__version__)
assert torch.cuda.is_available(), "CUDA required"
cap = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(0)} | cc={cap} | VRAM={vram_gb:.1f} GiB")
assert cap[0] >= 8, "GPU lacks native bf16 (T4?): switch runtime to A100 or L4"

In [ ]:
# ---- Mount Drive, unpack data, verify all restored .pt checkpoints --------------------------------------------------------
import os
import shutil
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

D = None
for cand in DRIVE_CANDIDATES:
    p = Path(cand)
    if (p / "t4_data.zip").exists() or (p / "out").exists():
        D = p
        break
if D is None:
    my_drive = Path("/content/drive/MyDrive")
    for p in sorted(my_drive.glob("*cpe342*")):
        if (p / "t4_data.zip").exists() or (p / "out").exists():
            D = p
            break
assert D is not None and (D / "t4_data.zip").exists(), f"t4_data.zip not found under {DRIVE_CANDIDATES}"
print("Using Drive folder:", D)

R = Path("/content/cpe342")
DATA = R / "04_image_classification/_download/task4_image_classification"
if not (DATA / "test.csv").exists() or len(os.listdir(DATA / "train")) < 31546 or len(os.listdir(DATA / "test")) < 25889:
    R.mkdir(parents=True, exist_ok=True)
    shutil.copy(D / "t4_data.zip", "/content/t4_data.zip")
    zipfile.ZipFile("/content/t4_data.zip").extractall(R)
    os.remove("/content/t4_data.zip")
print("Unpacked images -> train:", len(os.listdir(DATA / "train")), "| test:", len(os.listdir(DATA / "test")))
OUT_PREDS = D / "out" / "preds"
OUT_MODELS = D / "out" / "models"
OUT_R30 = D / "out" / "r30_views"
OUT_PREDS.mkdir(parents=True, exist_ok=True)

needed_ckpts = [OUT_MODELS / f"t4_{TAG}_fold{f}.pt" for f in FOLDS]
if EXTRACT_TEST:
    needed_ckpts.append(OUT_MODELS / f"t4_{TAG}_full.pt")
missing = [p.name for p in needed_ckpts if not p.exists()]
assert not missing, f"Missing checkpoints in {OUT_MODELS}: {missing}"
print("Verified checkpoints in Drive models/:", [p.name for p in needed_ckpts])

In [ ]:
# ---- Run Multi-Res TTA (OOF folds + optional test full+5fold) -------------------------------------------------------------
import gc, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn.functional as F
import timm

NATIVE_HW = (120, 230)
MEAN = torch.tensor((0.485, 0.456, 0.406), device="cuda").view(1, 3, 1, 1)
STD = torch.tensor((0.229, 0.224, 0.225), device="cuda").view(1, 3, 1, 1)

def load_uint8(paths, n_threads=8):
    arr = np.empty((len(paths), *NATIVE_HW, 3), dtype=np.uint8)
    def _load(i):
        im = Image.open(paths[i]).convert("RGB")
        if im.size != (NATIVE_HW[1], NATIVE_HW[0]):
            im = im.resize((NATIVE_HW[1], NATIVE_HW[0]), Image.BILINEAR)
        arr[i] = np.asarray(im)
    with ThreadPoolExecutor(n_threads) as ex:
        list(ex.map(_load, range(len(paths))))
    return arr

def to_gpu_batch(arr_u8, hw):
    x = torch.from_numpy(arr_u8).cuda(non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
    return F.interpolate(x, size=hw, mode="bilinear", align_corners=False)

def zoom(x, factor=0.7):
    th = torch.zeros(x.shape[0], 2, 3, device=x.device)
    th[:, 0, 0], th[:, 1, 1] = factor, factor
    return F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False),
                         mode="bilinear", padding_mode="border", align_corners=False)

def centre_grey(x):
    h, w = x.shape[2:]
    m = torch.zeros(1, 1, h, w, device=x.device)
    m[..., h // 5:4 * h // 5, w // 5:4 * w // 5] = 1.0
    return x * m + 0.5 * (1.0 - m)

def load_dinov3_dynamic(ckpt_path: Path):
    model = timm.create_model(MODEL, pretrained=False, num_classes=5, dynamic_img_size=True)
    state = torch.load(ckpt_path, map_location="cuda")
    model.load_state_dict(state, strict=True)
    return model.to(device="cuda", memory_format=torch.channels_last).eval()

@torch.no_grad()
def predict_multires_views(model, paths, views):
    out = {f"{v}_{rk}": [] for v in views for rk in RES_KEYS}
    for i in range(0, len(paths), 4096):
        arr = load_uint8(paths[i:i + 4096])
        for j in range(0, len(arr), BS):
            batch_u8 = arr[j:j + BS]
            x_base = to_gpu_batch(batch_u8, BASE_HW) if any(v != "full" for v in views) else None
            view_192 = {}
            for v in views:
                if v == "zoom70":
                    view_192[v] = zoom(x_base, 0.7)
                elif v == "centre_grey":
                    view_192[v] = centre_grey(x_base)
            for hw, rk in zip(RESOLUTIONS, RES_KEYS):
                for v in views:
                    if v == "full":
                        xv = to_gpu_batch(batch_u8, hw)
                    else:
                        x192 = view_192[v]
                        xv = x192 if hw == BASE_HW else F.interpolate(x192, size=hw, mode="bilinear", align_corners=False)
                    xn = ((xv - MEAN) / STD).to(memory_format=torch.channels_last)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        p = 0.5 * (model(xn).float().softmax(-1) + model(xn.flip(-1)).float().softmax(-1))
                    out[f"{v}_{rk}"].append(p.cpu().numpy().astype(np.float32))
    return {k: np.concatenate(v, axis=0) for k, v in out.items()}

train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")
train_paths = [str(DATA / "train" / f) for f in train_df["file_name"]]
test_paths = [str(DATA / "test" / f) for f in test_df["file_name"]]
y = train_df["label"].astype(np.int64).values
fold_idx = [va for _, va in StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(y)), y)]
req_keys = [f"{v}_{rk}" for v in EVAL_VIEWS for rk in RES_KEYS]

# 1. Extract OOF folds (resume-safe)
t_all = time.time()
for f in FOLDS:
    va = fold_idx[f]
    out_path = OUT_PREDS / f"t4_mres_{TAG}_fold{f}.npz"
    if out_path.exists():
        with np.load(out_path) as z:
            if all(k in z.files and z[k].shape == (len(va), 5) for k in req_keys):
                print(f"fold {f}: valid cache on Drive -> skipping")
                continue
    t1 = time.time()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = load_dinov3_dynamic(OUT_MODELS / f"t4_{TAG}_fold{f}.pt")
    preds = predict_multires_views(model, [train_paths[idx] for idx in va], EVAL_VIEWS)
    peak = torch.cuda.max_memory_allocated() / 2**30

    parity_msg = ""
    r30_path = OUT_R30 / f"{TAG}_fold{f}.npz"
    if r30_path.exists():
        r30_full = np.load(r30_path)["full"]
        abs_diff = np.abs(preds["full_192x384"] - r30_full)
        mean_diff = float(abs_diff.mean())
        max_diff = float(abs_diff.max())
        agree = float(np.mean(preds["full_192x384"].argmax(1) == r30_full.argmax(1)))
        assert mean_diff < 2e-3 and agree >= 0.998 and max_diff < 0.08, (
            f"fold {f} 192x384 parity failed: mean={mean_diff:.2e}, max={max_diff:.4f}, agree={agree:.4f}"
        )
        parity_msg = f" | 192x384 parity mean={mean_diff:.2e}, max={max_diff:.4f}, agree={agree:.4f}"

    tmp = Path(f"/content/tmp_{out_path.name}")
    np.savez(tmp, val_idx=va, **preds)
    shutil.copy2(tmp, out_path.with_suffix(".tmp.npz"))
    os.replace(out_path.with_suffix(".tmp.npz"), out_path)
    tmp.unlink(missing_ok=True)
    print(f"fold {f}: saved {out_path.name} in {(time.time()-t1)/60:.1f}m | peak={peak:.2f} GiB{parity_msg}", flush=True)
    del model
    gc.collect()
    torch.cuda.empty_cache()

# 2. Extract test Multi-Res TTA (inference-only cache)
if EXTRACT_TEST:
    test_out = OUT_PREDS / f"t4_mres_{TAG}_test.npz"
    if test_out.exists():
        print(f"test cache {test_out.name} already exists on Drive -> skipping")
    else:
        fold_means = {f"fold_mean_{rk}": np.zeros((len(test_paths), 5), dtype=np.float64) for rk in RES_KEYS}
        for f in range(5):
            t1 = time.time()
            model = load_dinov3_dynamic(OUT_MODELS / f"t4_{TAG}_fold{f}.pt")
            preds = predict_multires_views(model, test_paths, ["full"])
            for rk in RES_KEYS:
                fold_means[f"fold_mean_{rk}"] += preds[f"full_{rk}"] / 5.0
            print(f"test fold {f}: done in {(time.time()-t1)/60:.1f}m", flush=True)
            del model
            gc.collect()
            torch.cuda.empty_cache()

        t1 = time.time()
        model = load_dinov3_dynamic(OUT_MODELS / f"t4_{TAG}_full.pt")
        full_preds = predict_multires_views(model, test_paths, ["full"])
        del model
        gc.collect()
        torch.cuda.empty_cache()
        print(f"test full refit: done in {(time.time()-t1)/60:.1f}m", flush=True)

        tmp = Path(f"/content/tmp_{test_out.name}")
        np.savez(
            tmp,
            **{k: v.astype(np.float32) for k, v in fold_means.items()},
            **{f"full_{rk}": full_preds[f"full_{rk}"].astype(np.float32) for rk in RES_KEYS},
            test_file_name=np.array(test_df["file_name"].tolist(), dtype="U"),
        )
        shutil.copy2(tmp, test_out.with_suffix(".tmp.npz"))
        os.replace(test_out.with_suffix(".tmp.npz"), test_out)
        tmp.unlink(missing_ok=True)
        print(f"saved {test_out.name} ({test_out.stat().st_size / 2**20:.2f} MB)")

print(f"Finished Multi-Res TTA in {(time.time()-t_all)/60:.1f}m")

In [ ]:
# ---- Summary of tiny files to pull back + auto-disconnect -----------------------------------------------------------------
for p in sorted(OUT_PREDS.glob(f"t4_mres_{TAG}*.npz")):
    print(f"  {p.name:36s} {p.stat().st_size / 2**20:6.2f} MB")

if AUTO_DISCONNECT:
    import time
    print(f"done -> ending runtime in 60 s [{time.strftime('%H:%M')} UTC]", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()